In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# split data into training and validation data, for both features and target
# The split is based on a random number generator. Supplying a numeric value to
# the random_state argument guarantees we get the same split every time we
# run this script.

melb_data = pd.read_csv("melb_data.csv")
melb_data.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


In [2]:
#Tagert variable
y = melb_data.Price
#Extract Features
features = ['Rooms', 'Bathroom', 'Landsize', 'Lattitude', 'Longtitude']
X = melb_data[features]

In [4]:
X_train,X_test,y_train,y_test = train_test_split(X,y)

In [5]:
#Define Model
melbourne_model = DecisionTreeRegressor(random_state=1)

#Fit Model
melbourne_model.fit(X_train,y_train)

DecisionTreeRegressor(random_state=1)

In [6]:
# get predicted prices on validation data

y_pred = melbourne_model.predict(X_test)
y_pred

array([1010000.,  570000., 2000000., ..., 1101500.,  500000.,  550000.])

In [7]:
#Validate the model. How accurate is the model? Use the Mean Absolute Error to see how close to the acctual values the predictions are
mae = mean_absolute_error(y_test,y_pred)
print(mae)

233689.3645557192


In [8]:
#The MAE value is a very big number meaning that the model is just useless can cannot be used in real world to predict house prices.
#Coz where on earth would you use a model that is off from the actual value by 1000's of dollars (5 figure values) to predict data?
# So you need to figure out how you can make the model better. 
#First idea that comes in mind, overfitting or underfiiting of the decision tree.
# How can I handle this scenario of fitting. I can handle it by making use of the max_leaf_nodes, when fitting the model

In [9]:
def get_mae(max_leaf_nodes):
    new_model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes,random_state=0)
    new_model.fit(X_train,y_train)
    y_pred= new_model.predict(X_test)

    mae = mean_absolute_error(y_test,y_pred)
    return mae

In [10]:
# compare MAE with differing values of max_leaf_nodes
print(f"Max Leaf Nodes: 5 \t {int(get_mae(5))}")
print(f"Max Leaf Nodes: 50 \t {int(get_mae(50))}")
print(f"Max Leaf Nodes: 500 \t {int(get_mae(500))}")
print(f"Max Leaf Nodes: 5000 \t {int(get_mae(5000))}")
print(f"Max Leaf Nodes: 50000 \t {int(get_mae(50000))}")
print(f"Max Leaf Nodes: 500000 \t {int(get_mae(500000))}")


Max Leaf Nodes: 5 	 348199
Max Leaf Nodes: 50 	 255980
Max Leaf Nodes: 500 	 210500
Max Leaf Nodes: 5000 	 228980
Max Leaf Nodes: 50000 	 230668
Max Leaf Nodes: 500000 	 230668


In [11]:
# compare MAE with differing values of max_leaf_nodes using a for Loop
for max_leaf_nodes in [5, 50, 500, 5000]:
    my_mae = get_mae(max_leaf_nodes)
    #print("Max leaf nodes: %d  \t\t Mean Absolute Error:  %d" %(max_leaf_nodes, my_mae))
    print(f"Max leaf nodes: {max_leaf_nodes} \t\t Mean Absolute Error: {int(my_mae)}")

Max leaf nodes: 5 		 Mean Absolute Error: 348199
Max leaf nodes: 50 		 Mean Absolute Error: 255980
Max leaf nodes: 500 		 Mean Absolute Error: 210500
Max leaf nodes: 5000 		 Mean Absolute Error: 228980


Write a loop that tries the following values for max_leaf_nodes from a set of possible values.

Call the get_mae function on each value of max_leaf_nodes. Store the output in some way that allows you to select the value of max_leaf_nodes that gives the most accurate model on your data.

In [12]:
candidate_max_leaf_nodes = [5, 25, 50, 100, 250, 500,5000]
# Write loop to find the ideal tree size from candidate_max_leaf_nodes
for max_leaf_nodes in [5, 25, 50, 100, 250, 500]:
    my_mae = get_mae(max_leaf_nodes)
   
# Store the best value of max_leaf_nodes (it will be either 5, 25, 50, 100, 250 or 500)

scores = {leaf_size: get_mae(leaf_size) for leaf_size in candidate_max_leaf_nodes}
best_tree_size = min(scores, key=scores.get)
best_tree_size

500

## Fit Model Using All Data
#### You know the best tree size. If you were going to deploy this model in practice, you would make it even more accurate by using all of the data and keeping that tree size. That is, you don't need to hold out the validation data now that you've made all your modeling decisions.

In [13]:
# Fit the model with best_tree_size. Fill in argument to make optimal size
final_model = DecisionTreeRegressor(max_leaf_nodes=best_tree_size, random_state=1)

# fit the final model
final_model.fit(X, y)

DecisionTreeRegressor(max_leaf_nodes=500, random_state=1)

In [14]:
predictions = final_model.predict(X)
predictions
mae = mean_absolute_error(y,predictions)
mae

156971.0248947204

In [15]:
# Generate a list of all values between 1 and 1000 (excluding 0, as it wouldn't make sense for max_leaf_nodes)
# Define the range of candidate max_leaf_nodes
# Define the range of candidate max_leaf_nodes
candidate_max_leaf_nodes = range(2, 1001)  # All values from 2 to 1000

# Write a loop to find the ideal tree size from candidate_max_leaf_nodes
scores = {}
for max_leaf_nodes in candidate_max_leaf_nodes:
    scores[max_leaf_nodes] = get_mae(max_leaf_nodes)

# Store the best value of max_leaf_nodes
best_tree_size = min(scores, key=scores.get)

# Output the best tree size
print(f"The ideal max_leaf_nodes is: {best_tree_size}")




The ideal max_leaf_nodes is: 659
